# Generate Per-Scene Frozen YOLOWorld Models

For each scene in `context_new.db` (365 scenes, Objects365 vocabulary), this notebook:

1. Loads a fine-tuned YOLOWorld checkpoint
2. Calls `model.set_classes(scene_classes)` to embed the scene's vocabulary into the model weights
3. Saves a **frozen per-scene model** with `model.save()`

The saved models have the text encoder baked in — they can be loaded on any machine **without** calling `set_classes()` at runtime, making them faster to start up and deployable on devices where CLIP is not available.

---
### Why compare?
| Mode | `set_classes()` at runtime | Frozen per-scene `.pt` |
|---|---|---|
| Model size | 1 shared model | N × scene model |
| Switch latency | CLIP encode + load (100-300 ms) | Just load weights |
| Disk usage | ~50 MB × 1 | ~50 MB × N scenes |
| Flexibility | Any class at any time | Fixed at save time |
| Best for | Dynamic / research | Deployed / edge device |

> **Tip:** Run the benchmark cell (§5) to measure the actual switch latency on your hardware.

## 1 — Configuration

In [ ]:
import os, sys
from pathlib import Path

# ─── Input model ──────────────────────────────────────────────────────────────
# Point this to your fine-tuned checkpoint.
# If you haven't fine-tuned yet, the base 'yolov8s-worldv2.pt' also works
# (Ultralytics auto-downloads it on first use).
MODEL_PATH = 'yolov8s-worldv2.pt'          # ← change to your fine-tuned .pt

# ─── Context DB ───────────────────────────────────────────────────────────────
# Use context_new.db (Objects365 vocab) or context.db (COCO-80 vocab)
DB_PATH = 'context_new.db'                  # ← adjust path if running from Colab

# ─── Output directory ─────────────────────────────────────────────────────────
OUT_DIR  = Path('scene_models')             # per-scene .pt files saved here
OUT_DIR.mkdir(exist_ok=True)

# ─── Optional: only generate for a subset of scenes ──────────────────────────
# None = all 365 scenes.  Set to a list of scene names to generate only those.
SCENE_FILTER = None
# Example: SCENE_FILTER = ['living_room', 'kitchen', 'office', 'street', 'park']

# ─── Skip if already generated ────────────────────────────────────────────────
SKIP_EXISTING = True

print('Config OK')
print(f'  Model  : {MODEL_PATH}')
print(f'  DB     : {DB_PATH}')
print(f'  Out dir: {OUT_DIR.resolve()}')

## 2 — Load Scene → Classes Mapping from DB

In [ ]:
import sqlite3, json

conn = sqlite3.connect(DB_PATH)
cur  = conn.cursor()
cur.execute('SELECT scene_name, yolo_classes FROM scene_context ORDER BY scene_name')
rows = cur.fetchall()
conn.close()

SCENES = {
    name: json.loads(cls_json)
    for name, cls_json in rows
    if (SCENE_FILTER is None or name in SCENE_FILTER)
}

print(f'Scenes loaded : {len(SCENES)}')
total_classes = sum(len(v) for v in SCENES.values())
print(f'Avg classes   : {total_classes / len(SCENES):.1f}')

# Preview a few
for name, cls in list(SCENES.items())[:5]:
    print(f'  {name:35s} ({len(cls):3d}) {cls[:4]}...')

## 3 — Generate Frozen Per-Scene Models

For each scene we:
1. Load the base checkpoint (once)
2. Call `model.set_classes(classes)` — writes CLIP text embeddings into the model
3. Call `model.save(path)` — serialises the frozen vocabulary into the `.pt` file

In [ ]:
import time
from ultralytics import YOLO
from tqdm.auto import tqdm

# Load base model once — we clone vocab state per scene
print(f'Loading base model: {MODEL_PATH} ...')
base_model = YOLO(MODEL_PATH)
print('Base model loaded.')

results_log = []   # (scene, num_classes, duration_s, output_path)
skipped = 0

for scene_name, classes in tqdm(SCENES.items(), desc='Generating scene models'):
    out_path = OUT_DIR / f'{scene_name}.pt'

    if SKIP_EXISTING and out_path.exists():
        skipped += 1
        continue

    t0 = time.perf_counter()

    # Re-load base weights fresh for each scene so set_classes() is isolated
    model = YOLO(MODEL_PATH)
    model.set_classes(classes)
    model.save(str(out_path))

    dt = time.perf_counter() - t0
    results_log.append((scene_name, len(classes), dt, str(out_path)))

print(f'\nGenerated : {len(results_log)} models')
print(f'Skipped   : {skipped} (already exist)')
if results_log:
    avg_t = sum(r[2] for r in results_log) / len(results_log)
    print(f'Avg time  : {avg_t:.2f} s/scene')

# Disk usage
total_mb = sum(p.stat().st_size for p in OUT_DIR.glob('*.pt')) / 1e6
print(f'Total disk: {total_mb:.1f} MB  ({len(list(OUT_DIR.glob("*.pt")))} files)')

## 4 — Verify: Load a Frozen Model and Run Inference

In [ ]:
from ultralytics import YOLO
import numpy as np

# Pick a scene to verify
test_scene = 'living_room'
frozen_path = OUT_DIR / f'{test_scene}.pt'

if not frozen_path.exists():
    print(f'No frozen model for {test_scene!r} — pick a scene that was generated above.')
else:
    frozen = YOLO(str(frozen_path))

    # Inspect embedded class names
    names = frozen.names   # dict {id: name}
    print(f'Scene      : {test_scene}')
    print(f'Classes    : {list(names.values())}')
    print(f'Num classes: {len(names)}')

    # Run on a blank image to confirm the model is healthy
    dummy = np.zeros((640, 640, 3), dtype=np.uint8)
    result = frozen.predict(dummy, verbose=False, conf=0.25)
    print(f'Inference  : OK ({len(result[0].boxes)} detections on blank image)')

## 5 — Benchmark: Dynamic `set_classes()` vs Frozen Model

Measures the full switch latency for both approaches over N scene transitions.

In [ ]:
import time, random, statistics
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt

# Scenes to benchmark (random subset so it finishes quickly)
BENCH_SCENES = random.sample(list(SCENES.keys()), min(20, len(SCENES)))
DUMMY_IMG    = np.zeros((640, 640, 3), dtype=np.uint8)
WARM_ROUNDS  = 2    # warm-up rounds (not counted)
BENCH_ROUNDS = 5    # timed rounds per scene


# ── A: Dynamic set_classes() on a single shared model ────────────────────────
print('Benchmarking Mode A: dynamic set_classes()...')
dyn_model = YOLO(MODEL_PATH)

dyn_switch_times = []
dyn_infer_times  = []

for scene in BENCH_SCENES:
    classes = SCENES[scene]
    # warm-up
    for _ in range(WARM_ROUNDS):
        dyn_model.set_classes(classes)
        dyn_model.predict(DUMMY_IMG, verbose=False)
    # timed
    for _ in range(BENCH_ROUNDS):
        t0 = time.perf_counter()
        dyn_model.set_classes(classes)
        t1 = time.perf_counter()
        dyn_model.predict(DUMMY_IMG, verbose=False)
        t2 = time.perf_counter()
        dyn_switch_times.append((t1 - t0) * 1000)
        dyn_infer_times.append((t2 - t1) * 1000)

print(f'  set_classes() : {statistics.mean(dyn_switch_times):.1f} ± {statistics.stdev(dyn_switch_times):.1f} ms')
print(f'  inference     : {statistics.mean(dyn_infer_times):.1f} ± {statistics.stdev(dyn_infer_times):.1f} ms')


# ── B: Load frozen per-scene model ───────────────────────────────────────────
frozen_scenes = [s for s in BENCH_SCENES if (OUT_DIR / f'{s}.pt').exists()]
if frozen_scenes:
    print(f'\nBenchmarking Mode B: frozen model load+infer ({len(frozen_scenes)} scenes)...')

    frz_load_times  = []
    frz_infer_times = []

    for scene in frozen_scenes:
        path = str(OUT_DIR / f'{scene}.pt')
        # warm-up
        for _ in range(WARM_ROUNDS):
            m = YOLO(path)
            m.predict(DUMMY_IMG, verbose=False)
        # timed
        for _ in range(BENCH_ROUNDS):
            t0 = time.perf_counter()
            m = YOLO(path)
            t1 = time.perf_counter()
            m.predict(DUMMY_IMG, verbose=False)
            t2 = time.perf_counter()
            frz_load_times.append((t1 - t0) * 1000)
            frz_infer_times.append((t2 - t1) * 1000)

    print(f'  model load    : {statistics.mean(frz_load_times):.1f} ± {statistics.stdev(frz_load_times):.1f} ms')
    print(f'  inference     : {statistics.mean(frz_infer_times):.1f} ± {statistics.stdev(frz_infer_times):.1f} ms')
else:
    print('\nNo frozen models found — run §3 first.')
    frz_load_times = frz_infer_times = []

In [ ]:
# ── Visualise benchmark results ───────────────────────────────────────────────
if frz_load_times:
    labels  = ['A: set_classes\n(switch only)', 'A: inference', 'B: model load', 'B: inference']
    means   = [
        statistics.mean(dyn_switch_times),
        statistics.mean(dyn_infer_times),
        statistics.mean(frz_load_times),
        statistics.mean(frz_infer_times),
    ]
    stdevs  = [
        statistics.stdev(dyn_switch_times),
        statistics.stdev(dyn_infer_times),
        statistics.stdev(frz_load_times),
        statistics.stdev(frz_infer_times),
    ]
    colors = ['#3b82f6', '#93c5fd', '#f59e0b', '#fcd34d']

    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.bar(labels, means, yerr=stdevs, capsize=6, color=colors, edgecolor='none', width=0.5)
    ax.set_ylabel('Time (ms)')
    ax.set_title('Dynamic set_classes() vs Frozen Per-Scene Model — switch + inference latency')
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                f'{mean:.0f} ms', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)
    plt.tight_layout()
    plt.savefig('benchmark_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: benchmark_results.png')
else:
    print('Run Mode B benchmark first.')

## 6 — Export Scene Models to ONNX / NCNN

Optional: convert frozen `.pt` models to ONNX or NCNN for deployment on Raspberry Pi.

In [ ]:
from ultralytics import YOLO
from pathlib import Path
from tqdm.auto import tqdm

# Which scenes to export (None = all generated scenes)
EXPORT_SCENES = None  # or e.g. ['living_room', 'kitchen', 'office']
EXPORT_FORMAT = 'onnx'   # 'onnx' | 'ncnn' | 'engine'
EXPORT_IMGSZ  = 640

pt_files = sorted(OUT_DIR.glob('*.pt'))
if EXPORT_SCENES:
    pt_files = [OUT_DIR / f'{s}.pt' for s in EXPORT_SCENES if (OUT_DIR / f'{s}.pt').exists()]

print(f'Exporting {len(pt_files)} models to {EXPORT_FORMAT.upper()}...')

for pt in tqdm(pt_files, desc='Export'):
    scene = pt.stem
    ext = {'onnx': '.onnx', 'ncnn': '_ncnn_model', 'engine': '.engine'}.get(EXPORT_FORMAT, '.onnx')
    out_check = pt.with_suffix(ext) if not EXPORT_FORMAT == 'ncnn' else pt.parent / (scene + ext)
    if out_check.exists():
        continue
    try:
        m = YOLO(str(pt))
        m.export(format=EXPORT_FORMAT, imgsz=EXPORT_IMGSZ, simplify=True)
    except Exception as e:
        print(f'  [WARN] {scene}: {e}')

print('Done.')

## 7 — Update context_new.db with Frozen Model Paths

Once you've copied the generated `scene_models/*.pt` files to your `backend/models/` directory, run this cell to update the `model_file` column in `context_new.db` so the backend automatically loads the right frozen model per scene.

In [ ]:
import sqlite3

# Set to True only after you've copied scene_models/ → backend/models/
UPDATE_DB = False

if UPDATE_DB:
    conn = sqlite3.connect(DB_PATH)
    cur  = conn.cursor()
    updated = 0
    for pt in OUT_DIR.glob('*.pt'):
        scene = pt.stem
        cur.execute(
            'UPDATE scene_context SET model_file=? WHERE scene_name=?',
            (pt.name, scene)
        )
        updated += cur.rowcount
    conn.commit()
    conn.close()
    print(f'Updated model_file for {updated} scenes in {DB_PATH}')
else:
    print('Set UPDATE_DB = True to write model paths to the database.')
    print('(Do this after copying scene_models/ to backend/models/)')

## 8 — Download Results (Colab)

Pack the generated scene models into a zip and download to your local machine.

In [ ]:
import shutil, os

zip_path = Path('scene_models.zip')
shutil.make_archive('scene_models', 'zip', OUT_DIR)
size_mb = zip_path.stat().st_size / 1e6
print(f'Packed: {zip_path}  ({size_mb:.1f} MB)')

try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print(f'Not in Colab — copy {zip_path} manually.')

# Also download the updated DB
try:
    from google.colab import files as cf
    cf.download(DB_PATH)
except ImportError:
    pass

---
## Appendix — How to use in the backend

### Option A — Dynamic (current default)
```
# context_new.db: model_file = 'yolov8s-worldv2.pt' for all scenes
# _switch_scene() calls set_classes() on the single shared model
```
Switch the backend to `context_new.db` in `context_manager.py`:
```python
self.db_path = os.path.join(base_dir, 'data', 'context_new.db')
```

### Option B — Frozen per-scene models
1. Copy `scene_models/` contents into `backend/models/`
2. Run §7 with `UPDATE_DB = True` to update `model_file` in `context_new.db`
3. In `app.py` set `_phase2_mode = 'model'` (or toggle via the research page)

Now `_switch_scene()` in **model mode** will hot-swap the full `.pt` for each scene.

### Hybrid (recommended)
Use frozen models for frequently-visited scenes (fast) and fall back to `set_classes()` for rare scenes.  
The `/manage-context` page lets you set `model_file` per scene individually.